In [3]:
from pathlib import Path 
import pandas as pd 
import zipfile 

primary_zip = Path('/home/labuser/Desktop/Persistent_Folder/Workspace/day-04-evidence/source-extract/Insurance Claims Fraud Data.zip')

source_note = None

if primary_zip.exists():
    source_note = f'primary zip: {primary_zip}'
    with zipfile.ZipFile(primary_zip) as z:
        names = [n for n in z.namelist() if not n.endswith('/')]
        print('files inside zip are:', names[:10])
        chosen = next((n for n in names if n.lower().endswith('.csv')), None)
        if chosen:
            with z.open(chosen) as f:
                df = pd.read_csv(f)
        else:
            chosen = next((n for n in names if n.lower().endswith('.xlsx', '.xls')), None)
            if chosen is None:
                raise ValueError('No CSV / XLSX / XLS files found')
            else:
                with z.open(chosen) as f:
                    df = pd.read_excel(f)
        print('chosen_file_inside_zip:', chosen)
else: 
    raise FileNotFoundError('File does not exist')
print('source_note:', source_note)
print('rows:', len(df))
print('columns', list(df.columns))
df.head(5)

files inside zip are: ['insurance_data.csv', 'vendor_data.csv']
chosen_file_inside_zip: insurance_data.csv
source_note: primary zip: /home/labuser/Desktop/Persistent_Folder/Workspace/day-04-evidence/source-extract/Insurance Claims Fraud Data.zip
rows: 10000
columns ['TXN_DATE_TIME', 'TRANSACTION_ID', 'CUSTOMER_ID', 'POLICY_NUMBER', 'POLICY_EFF_DT', 'LOSS_DT', 'REPORT_DT', 'INSURANCE_TYPE', 'PREMIUM_AMOUNT', 'CLAIM_AMOUNT', 'CUSTOMER_NAME', 'ADDRESS_LINE1', 'ADDRESS_LINE2', 'CITY', 'STATE', 'POSTAL_CODE', 'SSN', 'MARITAL_STATUS', 'AGE', 'TENURE', 'EMPLOYMENT_STATUS', 'NO_OF_FAMILY_MEMBERS', 'RISK_SEGMENTATION', 'HOUSE_TYPE', 'SOCIAL_CLASS', 'ROUTING_NUMBER', 'ACCT_NUMBER', 'CUSTOMER_EDUCATION_LEVEL', 'CLAIM_STATUS', 'INCIDENT_SEVERITY', 'AUTHORITY_CONTACTED', 'ANY_INJURY', 'POLICE_REPORT_AVAILABLE', 'INCIDENT_STATE', 'INCIDENT_CITY', 'INCIDENT_HOUR_OF_THE_DAY', 'AGENT_ID', 'VENDOR_ID']


,TXN_DATE_TIME,TRANSACTION_ID,CUSTOMER_ID,POLICY_NUMBER,POLICY_EFF_DT,LOSS_DT,REPORT_DT,INSURANCE_TYPE,PREMIUM_AMOUNT,CLAIM_AMOUNT,...,CLAIM_STATUS,INCIDENT_SEVERITY,AUTHORITY_CONTACTED,ANY_INJURY,POLICE_REPORT_AVAILABLE,INCIDENT_STATE,INCIDENT_CITY,INCIDENT_HOUR_OF_THE_DAY,AGENT_ID,VENDOR_ID
0,2020-06-01 00:00:00,TXN00000001,A00003822,PLC00008468,2015-06-23,2020-05-16,2020-05-21,Health,157.13,9000,...,A,Major Loss,Police,0,1,GA,Savannah,4,AGENT00413,VNDR00556
1,2020-06-01 00:00:00,TXN00000002,A00008149,PLC00009594,2018-04-21,2020-05-13,2020-05-18,Property,141.71,26000,...,A,Total Loss,Ambulance,1,0,AL,Montgomery,0,AGENT00769,VNDR00592
2,2020-06-01 00:00:00,TXN00000003,A00003172,PLC00007969,2019-10-03,2020-05-21,2020-05-26,Property,157.24,13000,...,A,Total Loss,Police,0,1,CO,Grand Junction,19,AGENT00883,VNDR00031
3,2020-06-01 00:00:00,TXN00000004,A00007572,PLC00009292,2016-11-29,2020-05-14,2020-05-19,Health,172.87,16000,...,A,Minor Loss,Ambulance,0,0,GA,Savannah,12,AGENT00278,VNDR00075
4,2020-06-01 00:00:00,TXN00000005,A00008173,PLC00000204,2011-12-26,2020-05-17,2020-05-22,Travel,88.53,3000,...,A,Major Loss,Police,0,1,TN,Nashville,18,AGENT00636,VNDR00472


In [4]:
import re

cols = list(df.columns)
lower_cols = { str(c).lower(): c for c in cols }

def find_col(tokens):
    for c in cols:
        name = str(c).lower()
        if any(t in name for t in tokens):
            return c
    return None 

claim_id_col = find_col(['claim', 'id']) or cols[0]
status_col = find_col(['claim_status', 'decision', 'approved', 'fraud'])
amount_col = find_col(['claim_amount', 'claim', 'paid'])
region_col = find_col(['region', 'state', 'city', 'location'])

semantic_metric = {
    'metric_name': 'claim_approval_rate',
    'formula': 'approved_claims / reviewed_claims',
    'claim_id_col_guess': claim_id_col,
    'status_col_guess': status_col,
    'amount_col_guess': amount_col,
    'region_col_guess': region_col,
    'needs_business_confirmation': True
}

semantic_metric



{'metric_name': 'claim_approval_rate',
 'formula': 'approved_claims / reviewed_claims',
 'claim_id_col_guess': 'TRANSACTION_ID',
 'status_col_guess': 'CLAIM_STATUS',
 'amount_col_guess': 'CLAIM_AMOUNT',
 'region_col_guess': 'CITY',
 'needs_business_confirmation': True}

In [5]:
if status_col:
    status_values = df[status_col].astype(str).str.lower()
    approved_mask = status_values.str.contains('approved|approve|accepted|settled|paid|yes|1|a', regex=True, na=False)
    reviewed_mask = status_values.notna()
else:
    approved_mask = pd.Series([False] * len(df))
    reviewed_mask = pd.Series([True] * len(df))

reviewed_claims = int(reviewed_mask.sum())
approved_claims = int(approved_mask.sum())

approval_rate = round(approved_claims / reviewed_claims, 4) if reviewed_claims else None 

metric_result = pd.DataFrame([{
    'metric_name': 'claim_approval_rate',
    'approved_claims': approved_claims,
    'reviewed_claims': reviewed_claims,
    'approval_rate': approval_rate,
    'status_column_used': status_col,
    'needs_business_confirmation':  True
}])

metric_result

,metric_name,approved_claims,reviewed_claims,approval_rate,status_column_used,needs_business_confirmation
0,claim_approval_rate,9497,10000,0.9497,CLAIM_STATUS,True


In [6]:
import sqlite3
from datetime import datetime

safe_df = df.copy()
safe_df.columns = [re.sub(r'[^0-9a-zA-Z_]+', '_', str(c)).strip('_').lower() or f'col_{i}' for i,  c in enumerate(safe_df.columns)]

conn = sqlite3.connect(':memory:')

safe_df.to_sql('claims_data', conn, index=False, if_exists='replace')


pii_tokens = ['name', 'email', 'phone', 'mobile', 'address', 'dob', 'birth', 'ssn', 'pan']
pii_columns = [ c for c in safe_df.columns if any(tok in c.lower() for tok in pii_tokens)]

query_budget = {
    'max_rows_returned': 20,
    'max_sql_characters': 1200,
    'allow_write_operations': False,
}

def inspect_sql(sql):
    lowered = sql.lower().strip()
    denied_reasons = []
    if not lowered.startswith('select'):
        denied_reasons.append('only SELECT queries are allowed')
    for token in ['insert', 'update', 'delete', 'drop', 'alter', 'create', 'truncate']:
        if token in lowered:
            denied_reasons.append(f'Write/DDL operation blocked: {token.strip()}')
    if 'select *' in lowered:
        denied_reasons.append('SELECT * is blocked')
    for col in pii_columns:
        if col.lower() in lowered:
            denied_reasons.append(f'PII-like columns are blocked: {col}')
    if len(sql) > query_budget['max_sql_characters']:
        denied_reasons.append('SQL exceeds the charcter budget')
    if 'limit' not in lowered and 'count(' not in lowered and 'avg(' not in lowered and 'sum(' not in lowered:
        denied_reasons.append('Row-level query must include LIMIT or aggregate')
    return denied_reasons


def governed_query(sql, user='claims_analyst_training'):
    reasons = inspect_sql(sql)
    allowed = len(reasons) == 0
    rows = None
    result_preview = None
    if allowed:
        result = pd.read_sql_query(sql, conn)
        rows = len(result)
        if rows > query_budget['max_rows_returned']:
            allowed = False
            reasons.append('Result row budget exceeded')
            result_preview = None
        else:
            result_preview = result
    audit = {
        'timestamp': datetime.utcnow().isoformat(),
        'user': user,
        'allowed': allowed,
        'denied_reasons': '; '.join(reasons),
        'sql': sql,
        'rows_returned': rows,
        'pii_columns_blocked': '; '.join(pii_columns),
    }
    return allowed, result_preview, audit

print('safe_table_columns:', list(safe_df.columns))
print('pii_columns_detected:', pii_columns)





safe_table_columns: ['txn_date_time', 'transaction_id', 'customer_id', 'policy_number', 'policy_eff_dt', 'loss_dt', 'report_dt', 'insurance_type', 'premium_amount', 'claim_amount', 'customer_name', 'address_line1', 'address_line2', 'city', 'state', 'postal_code', 'ssn', 'marital_status', 'age', 'tenure', 'employment_status', 'no_of_family_members', 'risk_segmentation', 'house_type', 'social_class', 'routing_number', 'acct_number', 'customer_education_level', 'claim_status', 'incident_severity', 'authority_contacted', 'any_injury', 'police_report_available', 'incident_state', 'incident_city', 'incident_hour_of_the_day', 'agent_id', 'vendor_id']
pii_columns_detected: ['customer_name', 'address_line1', 'address_line2', 'ssn']


In [7]:
allowed_sql = "SELECT COUNT(*) AS claim_rows FROM claims_data"
allowed, result, audit = governed_query(allowed_sql)
print('Allowed QUERY RESULT')
print(result)
print('AUDIT')
print(audit)



blocked_sql = "SELECT *  FROM claims_data LIMIT 5"
allowed_b, result_b, audit_b = governed_query(blocked_sql)
print('\nBLOCKED QUERY RESULT')
print(result_b)
print('AUDIT')
print(audit_b)

Allowed QUERY RESULT
   claim_rows
0       10000
AUDIT
{'timestamp': '2026-07-30T10:52:38.631395', 'user': 'claims_analyst_training', 'allowed': True, 'denied_reasons': '', 'sql': 'SELECT COUNT(*) AS claim_rows FROM claims_data', 'rows_returned': 1, 'pii_columns_blocked': 'customer_name; address_line1; address_line2; ssn'}

BLOCKED QUERY RESULT
None
AUDIT
{'timestamp': '2026-07-30T10:52:38.633404', 'user': 'claims_analyst_training', 'allowed': False, 'denied_reasons': 'SELECT * is blocked', 'sql': 'SELECT *  FROM claims_data LIMIT 5', 'rows_returned': None, 'pii_columns_blocked': 'customer_name; address_line1; address_line2; ssn'}


/tmp/ipykernel_48005/3999412087.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().isoformat(),
/tmp/ipykernel_48005/3999412087.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().isoformat(),


In [8]:
user_entitlements = {
    'claims_analyst_training' : {
        'allowed_region': None,
        'can_view_pii': False,
        'max_rows': 20,
    },
    'data_steward_training' : {
        'allowed_region': None,
        'can_view_pii': True,
        'max_rows': 20,
    },
}

permission_note = pd.DataFrame([
    {'user': user, **rules} for user, rules in user_entitlements.items()
])

permission_note

,user,allowed_region,can_view_pii,max_rows
0,claims_analyst_training,None,False,20
1,data_steward_training,None,True,20


In [12]:
policy_docs = pd.DataFrame([
    {
        'source_id': 'DOC-CLAIMS-001',
        'title': 'Claims review policy',
        'text': 'Claims with missing documents require manual review before approval. Fraud indicators must be reviewed by an authorized analyst.'
    },
    {
        'source_id': 'DOC-PII-002',
        'title': 'PII Serving policy',
        'text': 'AI assistants may receive aggregated or masked claims evidence. Raw name, phone, email, address and date of birth must not be exposed byb default.'
    },
    {
        'source_id': 'DOC-AUDIT-003',
        'title': 'Audit policy',
        'text': 'Every natural-language data access request must log user, question, generated SQL or retreival plan, permission decision, sources and result size'
    },
])

def keyword_retrieve(question, docs, top_k=2):
    q_terms = set(re.findall(r'[a-zA-Z]+', question.lower()))
    scored = []
    for _, row in docs.iterrows():
        d_terms = set(re.findall(r'[a-zA-Z]+', row['text'].lower() + ' ' + row['title'].lower()))
        score = len(q_terms & d_terms)
        scored.append({ **row.to_dict(), 'score': score })
    return pd.DataFrame(scored).sort_values('score', ascending=False).head(top_k)

rag_question = 'Can the claims AI answer approval-rate questions and cite the control policy?'
retreived_docs = keyword_retrieve(rag_question, policy_docs)

retreived_docs


rag_answer = {
    'question': rag_question,
    'answer': retreived_docs,
    'citations': retreived_docs[['source_id', 'title']].to_dict('records'),
    'faithfulness_check': 'Answer is supported by retirved policy chunks'
}

rag_answer


{'question': 'Can the claims AI answer approval-rate questions and cite the control policy?',
 'answer':         source_id                 title  \
 1     DOC-PII-002    PII Serving policy   
 0  DOC-CLAIMS-001  Claims review policy   
 
                                                 text  score  
 1  AI assistants may receive aggregated or masked...      4  
 0  Claims with missing documents require manual r...      3  ,
 'citations': [{'source_id': 'DOC-PII-002', 'title': 'PII Serving policy'},
  {'source_id': 'DOC-CLAIMS-001', 'title': 'Claims review policy'}],
 'faithfulness_check': 'Answer is supported by retirved policy chunks'}

In [13]:
graph_edges = pd.DataFrame([
    { 'from': 'Customer:CUST-1001', 'relation': 'owns', 'to': 'Policy:POL-101' },
    { 'from': 'Policy:POL-101', 'relation': 'has_claim', 'to': 'Claim:CLM-5001' },
    { 'from': 'Claim:CLM-5001', 'relation': 'controlled_by', 'to': 'Document:DOC-CLAIMS-001' },
    { 'from': 'Claim:CLM-5001', 'relation': 'audited_by', 'to': 'Document:DOC-AUDIT-003' },
]) 

graph_path = graph_edges.to_dict('records')

graph_answer = {
    'question': 'Which policy and policy document are connected to Claim CLM-5001 ?',
    'path': graph_path,
    'answer': 'Claim CLM-5001 is connected to policy PL-101, owned by customer CUST-1001, and contorlled by claims review and audit policies',
    'citations': ['DOC-CLAIMS-001', 'DOC-AUDIT-003']
}

graph_edges

,from,relation,to
0,Customer:CUST-1001,owns,Policy:POL-101
1,Policy:POL-101,has_claim,Claim:CLM-5001
2,Claim:CLM-5001,controlled_by,Document:DOC-CLAIMS-001
3,Claim:CLM-5001,audited_by,Document:DOC-AUDIT-003
